# Banking Operations — Exploratory Data Analysis

## Project 01 | Business Analytics Portfolio

### Objective
Explore a banking operations dataset to understand transaction volumes, processing times, errors, SLA breaches, product performance and regional patterns.

### Questions
- What does the dataset look like?
- Are there data-quality issues?
- Which products take the longest to process?
- Which products have more errors or SLA breaches?
- Do processing time and errors appear to be related?
- Are there any unusual revenue values?


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")


## 2. Load the Dataset

In [ ]:
df = pd.read_csv("../data/raw/banking_operations_raw.csv")

df.head()


## 3. Understand the Dataset

In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df.describe()


## 4. Check Data Quality

In [ ]:
# Check missing values
df.isnull().sum()


In [ ]:
# Check duplicate rows
df.duplicated().sum()


In [ ]:
# Check category values
print("Products:")
print(df["Product"].value_counts(dropna=False))

print("\nRegions:")
print(df["Region"].value_counts(dropna=False))

print("\nSLA Breach:")
print(df["SLA_Breach"].value_counts(dropna=False))


## 5. Clean the Data

There are a few simple inconsistencies in the categorical values. We will standardise them before analysis.

For missing Region values, we use `Unknown` rather than guessing the correct region.


In [ ]:
data = df.copy()

# Standardise product names
data["Product"] = data["Product"].replace({
    "Saving": "Savings",
    "Loans": "Loan"
})

# Standardise region names
data["Region"] = data["Region"].replace({
    "U.K.": "UK"
})

# Keep missing regions visible
data["Region"] = data["Region"].fillna("Unknown")

# Convert processing time from text to minutes
data["Processing_Time_Min"] = (
    data["Processing_Time"].str.extract(r"(\d+)")[0].astype(int)
)

# Convert SLA breach to 0/1
data["SLA_Breach_Flag"] = data["SLA_Breach"].map({
    "Yes": 1,
    "No": 0
})

# Flag negative revenue
data["Negative_Revenue_Flag"] = data["Revenue"] < 0

# Remove duplicate rows
data = data.drop_duplicates()

data.head()


## 6. Explore the Products

In [ ]:
data["Product"].value_counts()


In [ ]:
plt.figure(figsize=(8, 5))

data["Product"].value_counts().plot(kind="bar")

plt.title("Number of Transactions by Product")
plt.xlabel("Product")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=20)
plt.show()


## 7. Processing Time by Product

In [ ]:
data.groupby("Product")["Processing_Time_Min"].mean().sort_values(ascending=False)


In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=data,
    x="Product",
    y="Processing_Time_Min",
    estimator="mean"
)

plt.title("Average Processing Time by Product")
plt.xlabel("Product")
plt.ylabel("Average Processing Time (minutes)")
plt.xticks(rotation=20)
plt.show()


## 8. Errors by Product

In [ ]:
data.groupby("Product")["Error_Count"].mean().sort_values(ascending=False)


In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=data,
    x="Product",
    y="Error_Count",
    estimator="mean"
)

plt.title("Average Error Count by Product")
plt.xlabel("Product")
plt.ylabel("Average Errors")
plt.xticks(rotation=20)
plt.show()


## 9. SLA Breaches

In [ ]:
data.groupby("Product")["SLA_Breach_Flag"].mean().sort_values(ascending=False) * 100


In [ ]:
plt.figure(figsize=(8, 5))

sla_rate = data.groupby("Product")["SLA_Breach_Flag"].mean() * 100

sla_rate.plot(kind="bar")

plt.title("SLA Breach Rate by Product")
plt.xlabel("Product")
plt.ylabel("SLA Breach Rate (%)")
plt.xticks(rotation=20)
plt.show()


## 10. Regional Analysis

In [ ]:
data.groupby("Region")["Processing_Time_Min"].mean().sort_values(ascending=False)


In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    data=data,
    x="Region",
    y="Processing_Time_Min",
    estimator="mean"
)

plt.title("Average Processing Time by Region")
plt.xlabel("Region")
plt.ylabel("Average Processing Time (minutes)")
plt.show()


## 11. Processing Time and Errors

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=data,
    x="Processing_Time_Min",
    y="Error_Count",
    hue="Product"
)

plt.title("Processing Time vs Error Count")
plt.xlabel("Processing Time (minutes)")
plt.ylabel("Error Count")
plt.show()


## 12. Revenue Check

In [ ]:
data[data["Revenue"] < 0][
    ["Transaction_ID", "Product", "Region", "Revenue", "Cost_Per_Txn"]
]


The negative revenue value is flagged for investigation rather than removed.

The dataset does not explain whether negative revenue represents a refund, reversal, chargeback or data-entry issue, so changing the value would require an unsupported assumption.


## 13. Key Findings

### Key observations

1. **Mortgage has the longest average processing time** and appears to be the most operationally complex product in the sample.
2. **Mortgage also has the highest average error count**, suggesting that longer processing may be associated with greater operational complexity.
3. **SLA breaches are concentrated in the higher-processing-time products.**
4. **Regional processing times differ**, although the dataset is too small to draw strong conclusions about regional performance.
5. **A negative revenue transaction exists** and should be investigated before using revenue data for profitability reporting.
6. The dataset also contains **inconsistent category labels and missing region information**, demonstrating the importance of data-quality checks before analysis.


## 14. Business Recommendations

- Investigate the Mortgage process to identify the steps causing longer processing times and higher error counts.
- Review SLA-breaching transactions to understand the operational reasons behind the delays.
- Standardise product and region values at the source to reduce future data-cleaning effort.
- Investigate the negative revenue transaction before using the dataset for financial reporting.
- With a larger dataset, analyse processing time by workflow stage, team and transaction complexity.


## 15. What I Learned

This project covers the foundations of Exploratory Data Analysis:

- Loading a dataset with Pandas
- Understanding rows, columns and data types
- Checking missing values and duplicates
- Cleaning categorical data
- Creating simple derived columns
- Using `groupby()` for business analysis
- Creating basic charts with Matplotlib and Seaborn
- Looking for relationships between variables
- Translating analysis into business observations

### Next progression

Future EDA projects will build on these foundations with more advanced statistical analysis, stronger visual storytelling, deeper anomaly analysis and more sophisticated business questions.
